# Model Card Generator - Lesson 5: AI Model Governance

**Scenario**: RetailGenius — an e-commerce company managing 8 AI models across departments

This solution notebook demonstrates:
1. Defining structured metadata for 3 representative AI models (M001, M002, M006)
2. Generating comprehensive Model Cards in Markdown format
3. Implementing model lifecycle tracking with pandas
4. Identifying overdue reviews and creating governance dashboards

---

### Quick Reference: Key Concepts

**Model Card Sections (Google Standard)**:
1. Model Details — ID, name, version, type, framework, developers, dates, license
2. Intended Use — Primary use cases, users, out-of-scope uses, demographic considerations
3. Training Data — Sources, size, preprocessing, quality issues, train/test split
4. Performance Metrics — Task-specific metrics, latency, A/B results
5. Ethical Considerations — Bias testing, known issues, demographic variation, fairness metrics, mitigation
6. Limitations — Known limitations, failure modes, edge cases
7. Recommendations & Monitoring — Monitoring plan, update schedule, governance, contact, next review

**Model Lifecycle Stages**: Development → Staging → Production → Retired

**Risk Tiers**: Critical (life-safety/financial), High (significant business impact), Medium (moderate/internal), Minimal (low risk)

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import json
from typing import Dict, List, Any

print(f"Pandas version: {pd.__version__}")
print(f"Current date: {datetime.now().strftime('%Y-%m-%d')}")

## Part 1: Define Model Metadata

In [ ]:
# Model 1: Product Recommendation Chatbot
model_1_metadata = {
    "model_id": "M001",
    "model_name": "Product Recommendation Chatbot",
    "version": "2.3.1",
    "model_type": "Generative AI (Large Language Model)",
    "framework": "GPT-4 with Fine-Tuning",
    "developers": ["Sarah Chen", "ML Team"],
    "department": "E-Commerce",
    "date_created": "2023-06-15",
    "last_updated": "2024-12-15",
    "intended_use": "Real-time product recommendations via conversational interface on e-commerce platform",
    "primary_users": "Retail customers browsing e-commerce website",
    "out_of_scope_uses": [
        "Price negotiation",
        "Customer support (non-product)",
        "Legal/financial advice",
        "Medical recommendations"
    ],
    "training_data_sources": [
        "Product catalog (50K products)",
        "Historical customer purchase data",
        "Search logs",
        "Product reviews"
    ],
    "training_data_size": "~2M customer interactions, 50K product records, 100K reviews",
    "preprocessing": [
        "Text cleaning",
        "Tokenization",
        "Product embeddings",
        "Removal of PII",
        "Handling missing values"
    ],
    "data_quality_issues": "Some products with sparse review data; older reviews may contain biased language",
    "performance_metrics": {
        "bleu_score": "0.72 (relevance of recommendations)",
        "ctr_improvement": "+18% vs baseline",
        "latency_p50": "450ms",
        "latency_p95": "800ms",
        "revenue_improvement": "+12%",
        "satisfaction_improvement": "+8%",
        "coverage": "95% of queries return recommendations"
    },
    "bias_testing": {
        "tests_conducted": [
            "Gender bias analysis",
            "Racial/ethnic representation",
            "Socioeconomic bias",
            "Age-based patterns"
        ],
        "known_issues": [
            "Slight over-recommendation of premium products",
            "Underrepresentation of niche/indie brands"
        ],
        "demographic_variation": {
            "gender": "Within 3% recall across M/F/Other",
            "age": "Within 5% recall across age groups"
        },
        "fairness_metrics": {
            "disparate_impact_ratio": "0.92 (acceptable)",
            "prediction_parity": "0.89"
        }
    },
    "mitigation_strategies": [
        "Undersampling premium products in training",
        "Explicit inclusion of diverse brand categories",
        "Periodic rebalancing"
    ],
    "limitations": [
        "Cannot explain reasoning for recommendations",
        "Performance degrades with new/niche products (cold-start problem)",
        "May amplify popular products (popularity bias)",
        "Language quality depends on input clarity"
    ],
    "failure_modes": [
        "Returns generic recommendations for ambiguous queries",
        "May recommend out-of-stock items",
        "Struggles with multi-modal requests"
    ],
    "edge_cases": [
        "New customer segments not seen in training",
        "Seasonal category shifts"
    ],
    "monitoring_plan": "Weekly CTR and BLEU score tracking; Monthly fairness metrics; Quarterly bias drift analysis",
    "update_schedule": "Re-train quarterly; Emergency retraining if performance drops >5%",
    "owner": "Sarah Chen",
    "next_review_date": "2025-03-15"
}

# Model 2: Demand Forecasting
model_2_metadata = {
    "model_id": "M002",
    "model_name": "Demand Forecasting",
    "version": "1.8.0",
    "model_type": "Time Series ML",
    "framework": "ARIMA + XGBoost Ensemble",
    "developers": ["Marcus Johnson", "Data Science Team"],
    "department": "Supply Chain",
    "date_created": "2022-03-20",
    "last_updated": "2024-11-28",
    "intended_use": "Predict product demand for inventory optimization",
    "primary_users": "Supply chain managers and inventory planners",
    "out_of_scope_uses": [
        "Real-time price adjustment",
        "Marketing campaign planning",
        "Customer behavior prediction beyond demand"
    ],
    "training_data_sources": [
        "24 months of historical sales data",
        "Seasonal indicators",
        "Marketing campaign calendar",
        "Inventory levels"
    ],
    "training_data_size": "24 months × 5,000 SKUs = 120K time series",
    "preprocessing": [
        "Detrending",
        "Seasonal decomposition",
        "Outlier detection and smoothing",
        "Feature engineering for lag variables"
    ],
    "data_quality_issues": "Some SKUs have gaps during stockouts; promotional spikes create anomalies",
    "performance_metrics": {
        "mape": "8.3% (Mean Absolute Percentage Error)",
        "rmse": "45 units",
        "forecast_accuracy_30d": "92.1%",
        "forecast_accuracy_90d": "87.5%"
    },
    "bias_testing": {
        "tests_conducted": [
            "Category performance bias",
            "Regional demand variation",
            "Seasonal fairness"
        ],
        "known_issues": [
            "Underforecasting for niche products",
            "Overforecasting during holiday season"
        ],
        "demographic_variation": {
            "product_category": "Within 12% accuracy across all categories",
            "region": "Within 8% accuracy across geographic regions"
        },
        "fairness_metrics": {
            "category_variation": "±12% across categories",
            "regional_variation": "±8% across regions"
        }
    },
    "mitigation_strategies": [
        "Category-specific models for niche products",
        "Holiday adjustment factors",
        "Regular retraining with latest data"
    ],
    "limitations": [
        "Cannot predict unprecedented events (e.g., supply chain disruptions)",
        "Accuracy degrades for new products",
        "Relies on historical patterns that may change"
    ],
    "failure_modes": [
        "Zero forecast for discontinued products",
        "Sudden spikes due to marketing campaigns"
    ],
    "edge_cases": [
        "New product launches",
        "Competitor promotions",
        "Supply shortages"
    ],
    "monitoring_plan": "Weekly accuracy checks; Monthly recalibration; Quarterly performance review",
    "update_schedule": "Monthly retraining; Quarterly model validation",
    "owner": "Marcus Johnson",
    "next_review_date": "2025-02-28"
}

# Model 3: Fraud Detection System
model_3_metadata = {
    "model_id": "M006",
    "model_name": "Fraud Detection System",
    "version": "4.2.0",
    "model_type": "Anomaly Detection",
    "framework": "Isolation Forest + Neural Network",
    "developers": ["David Lee", "Security Team"],
    "department": "Security",
    "date_created": "2021-09-10",
    "last_updated": "2024-12-05",
    "intended_use": "Detect fraudulent transactions and chargebacks in real-time",
    "primary_users": "Fraud analysts, transaction processing systems",
    "out_of_scope_uses": [
        "Customer credit scoring",
        "Behavioral profiling",
        "Identity verification"
    ],
    "training_data_sources": [
        "12 months of transaction data",
        "Chargeback reports",
        "Dispute resolution records",
        "Device fingerprinting data"
    ],
    "training_data_size": "500M transactions, 50K confirmed fraud cases",
    "preprocessing": [
        "PII anonymization",
        "Feature engineering for transaction patterns",
        "Handling class imbalance (0.01% fraud rate)",
        "Normalization of amounts across currencies"
    ],
    "data_quality_issues": "Severe class imbalance; some fraud cases not yet labeled",
    "performance_metrics": {
        "precision": "0.89",
        "recall": "0.76",
        "f1_score": "0.82",
        "auc_roc": "0.94",
        "avg_latency": "120ms",
        "false_positive_rate": "2.1%"
    },
    "bias_testing": {
        "tests_conducted": [
            "Geographic fairness analysis",
            "Customer demographics impact",
            "Transaction amount distribution"
        ],
        "known_issues": [
            "Higher false positive rates for international transactions",
            "Age group 65+ flagged at higher rate"
        ],
        "demographic_variation": {
            "geographic": "Within 3.2% false positive rate across regions",
            "age_group": "Within 4.8% false positive rate across age groups"
        },
        "fairness_metrics": {
            "geographic_variance": "±3.2%",
            "demographic_variance": "±4.8%"
        }
    },
    "mitigation_strategies": [
        "Geographic calibration of thresholds",
        "Age-aware fraud scoring",
        "Regular fairness audits"
    ],
    "limitations": [
        "Cannot detect never-before-seen fraud patterns",
        "High false positive rate impacts customer experience",
        "Requires labeled fraud data for retraining"
    ],
    "failure_modes": [
        "False positives block legitimate transactions",
        "Sophisticated fraud may bypass anomaly detection"
    ],
    "edge_cases": [
        "Large legitimate purchases",
        "International business travel",
        "New customer first transaction"
    ],
    "monitoring_plan": "Real-time fraud rate monitoring; Daily false positive analysis; Monthly performance review",
    "update_schedule": "Weekly retrain with new fraud data; Quarterly model evaluation",
    "owner": "David Lee",
    "next_review_date": "2025-03-05"
}

models_metadata = [model_1_metadata, model_2_metadata, model_3_metadata]

print(f"✓ Loaded metadata for {len(models_metadata)} models")
for model in models_metadata:
    print(f"  - {model['model_id']}: {model['model_name']}")

## Part 2: Model Card Generator Function

In [ ]:
def generate_model_card_markdown(metadata: Dict[str, Any]) -> str:
    """
    Generate a comprehensive Model Card in Markdown format from structured metadata.
    
    Args:
        metadata: Dictionary containing model information
        
    Returns:
        Markdown string formatted as a complete Model Card
    """
    
    card = []
    
    # Title
    card.append(f"# Model Card: {metadata['model_name']}\n")
    card.append(f"**Model ID**: {metadata['model_id']} | **Version**: {metadata['version']}\n")
    
    # Model Details Section
    card.append("## Model Details\n")
    card.append(f"- **Model Name**: {metadata['model_name']}")
    card.append(f"- **Model ID**: {metadata['model_id']}")
    card.append(f"- **Version**: {metadata['version']}")
    card.append(f"- **Model Type**: {metadata['model_type']}")
    card.append(f"- **Framework**: {metadata['framework']}")
    card.append(f"- **Developers**: {', '.join(metadata['developers'])}")
    card.append(f"- **Department**: {metadata['department']}")
    card.append(f"- **Date Created**: {metadata['date_created']}")
    card.append(f"- **Last Updated**: {metadata['last_updated']}")
    card.append(f"- **Owner**: {metadata['owner']}\n")
    
    # Intended Use Section
    card.append("## Intended Use\n")
    card.append(f"### Primary Purpose")
    card.append(f"{metadata['intended_use']}\n")
    card.append(f"### Primary Users")
    card.append(f"{metadata['primary_users']}\n")
    card.append(f"### Out-of-Scope Uses")
    for use in metadata['out_of_scope_uses']:
        card.append(f"- {use}")
    card.append("")
    
    # Training Data Section
    card.append("## Training Data\n")
    card.append(f"### Data Sources")
    for source in metadata['training_data_sources']:
        card.append(f"- {source}")
    card.append(f"\n### Data Size")
    card.append(f"{metadata['training_data_size']}\n")
    card.append(f"### Preprocessing")
    for step in metadata['preprocessing']:
        card.append(f"- {step}")
    card.append(f"\n### Data Quality Issues")
    card.append(f"{metadata['data_quality_issues']}\n")
    
    # Performance Metrics Section
    card.append("## Performance Metrics\n")
    for metric_name, metric_value in metadata['performance_metrics'].items():
        metric_display = metric_name.replace('_', ' ').title()
        card.append(f"- **{metric_display}**: {metric_value}")
    card.append("")
    
    # Ethical Considerations Section
    card.append("## Ethical Considerations & Bias Testing\n")
    card.append(f"### Bias Testing Conducted")
    for test in metadata['bias_testing']['tests_conducted']:
        card.append(f"- {test}")
    
    card.append(f"\n### Known Bias Issues")
    for issue in metadata['bias_testing']['known_issues']:
        card.append(f"- {issue}")
    
    card.append(f"\n### Demographic Performance Variation")
    for dimension, variation in metadata['bias_testing']['demographic_variation'].items():
        dimension_title = dimension.replace('_', ' ').title()
        card.append(f"- **{dimension_title}**: {variation}")
    
    card.append(f"\n### Fairness Metrics")
    for metric_name, metric_value in metadata['bias_testing']['fairness_metrics'].items():
        metric_display = metric_name.replace('_', ' ').title()
        card.append(f"- **{metric_display}**: {metric_value}")
    
    card.append(f"\n### Mitigation Strategies")
    for strategy in metadata['mitigation_strategies']:
        card.append(f"- {strategy}")
    card.append("")
    
    # Limitations Section
    card.append("## Limitations & Failure Modes\n")
    card.append(f"### Known Limitations")
    for limitation in metadata['limitations']:
        card.append(f"- {limitation}")
    
    card.append(f"\n### Failure Modes")
    for failure in metadata['failure_modes']:
        card.append(f"- {failure}")
    
    card.append(f"\n### Edge Cases")
    for edge_case in metadata['edge_cases']:
        card.append(f"- {edge_case}")
    card.append("")
    
    # Recommendations Section
    card.append("## Recommendations & Monitoring\n")
    card.append(f"### Monitoring Plan")
    card.append(f"{metadata['monitoring_plan']}\n")
    card.append(f"### Update Schedule")
    card.append(f"{metadata['update_schedule']}\n")
    card.append(f"### Next Review Date")
    card.append(f"{metadata['next_review_date']}\n")
    
    return "\n".join(card)

print("✓ Model Card generator function defined")

## Part 3: Generate Model Cards

In [ ]:
# Generate Model Cards for all 3 models
model_cards = {}

for model in models_metadata:
    model_id = model['model_id']
    model_cards[model_id] = generate_model_card_markdown(model)
    print(f"\n✓ Generated Model Card for {model_id}: {model['model_name']}")

print(f"\n{'='*80}")
print(f"Model Cards generated: {', '.join(model_cards.keys())}")
print(f"{'='*80}")

## Part 4: Display Generated Model Cards

In [ ]:
# Display Model Card for Product Recommendation Chatbot
print(model_cards['M001'])

In [ ]:
# Display Model Card for Demand Forecasting
print(model_cards['M002'])

In [ ]:
# Display Model Card for Fraud Detection
print(model_cards['M006'])

## Part 5: Model Lifecycle Tracking

In [ ]:
# Create comprehensive model inventory with lifecycle data
model_inventory = pd.DataFrame([
    {
        "Model ID": "M001",
        "Model Name": "Product Recommendation Chatbot",
        "Version": "2.3.1",
        "Owner": "Sarah Chen",
        "Lifecycle Stage": "Production",
        "Stage Entry Date": "2024-06-15",
        "Last Review Date": "2024-12-15",
        "Next Review Date": "2025-03-15",
        "Approval Status": "Approved",
        "Risk Tier": "Critical"
    },
    {
        "Model ID": "M002",
        "Model Name": "Demand Forecasting",
        "Version": "1.8.0",
        "Owner": "Marcus Johnson",
        "Lifecycle Stage": "Production",
        "Stage Entry Date": "2024-04-10",
        "Last Review Date": "2024-11-28",
        "Next Review Date": "2025-02-28",
        "Approval Status": "Approved",
        "Risk Tier": "High"
    },
    {
        "Model ID": "M003",
        "Model Name": "Image Search Engine",
        "Version": "3.1.2",
        "Owner": "Lisa Wang",
        "Lifecycle Stage": "Production",
        "Stage Entry Date": "2023-12-01",
        "Last Review Date": "2024-10-10",
        "Next Review Date": "2025-01-10",
        "Approval Status": "Approved",
        "Risk Tier": "High"
    },
    {
        "Model ID": "M004",
        "Model Name": "Customer Sentiment Analyzer",
        "Version": "1.5.0",
        "Owner": "James Rodriguez",
        "Lifecycle Stage": "Staging",
        "Stage Entry Date": "2025-01-20",
        "Last Review Date": "2025-01-20",
        "Next Review Date": "2025-04-20",
        "Approval Status": "Pending",
        "Risk Tier": "Medium"
    },
    {
        "Model ID": "M005",
        "Model Name": "Dynamic Pricing Optimizer",
        "Version": "2.0.1",
        "Owner": "Priya Patel",
        "Lifecycle Stage": "Production",
        "Stage Entry Date": "2023-10-15",
        "Last Review Date": "2024-09-30",
        "Next Review Date": "2024-12-30",
        "Approval Status": "Approved",
        "Risk Tier": "Critical"
    },
    {
        "Model ID": "M006",
        "Model Name": "Fraud Detection System",
        "Version": "4.2.0",
        "Owner": "David Lee",
        "Lifecycle Stage": "Production",
        "Stage Entry Date": "2024-05-20",
        "Last Review Date": "2024-12-05",
        "Next Review Date": "2025-03-05",
        "Approval Status": "Approved",
        "Risk Tier": "Critical"
    },
    {
        "Model ID": "M007",
        "Model Name": "Content Generator",
        "Version": "1.2.0",
        "Owner": "Elena Kowalski",
        "Lifecycle Stage": "Development",
        "Stage Entry Date": "2024-11-01",
        "Last Review Date": "2025-01-15",
        "Next Review Date": "2025-04-15",
        "Approval Status": "In Review",
        "Risk Tier": "High"
    },
    {
        "Model ID": "M008",
        "Model Name": "Customer Churn Predictor",
        "Version": "1.1.0",
        "Owner": "Robert Singh",
        "Lifecycle Stage": "Staging",
        "Stage Entry Date": "2025-01-10",
        "Last Review Date": "2025-01-10",
        "Next Review Date": "2025-04-10",
        "Approval Status": "Pending",
        "Risk Tier": "Medium"
    },
    {
        "Model ID": "M009",
        "Model Name": "Legacy Image Classifier v1",
        "Version": "1.0.5",
        "Owner": "Lisa Wang",
        "Lifecycle Stage": "Retired",
        "Stage Entry Date": "2023-08-01",
        "Last Review Date": "2025-02-10",
        "Next Review Date": "2025-02-10",
        "Approval Status": "Archived",
        "Risk Tier": "Medium"
    }
])

# Convert date columns to datetime
date_columns = ['Stage Entry Date', 'Last Review Date', 'Next Review Date']
for col in date_columns:
    model_inventory[col] = pd.to_datetime(model_inventory[col])

print("Model Inventory (9 models including 1 retired):")
print(model_inventory[['Model ID', 'Model Name', 'Lifecycle Stage', 'Approval Status', 'Risk Tier']])

## Part 6A: Approval Gates Check Function

In [ ]:
# Calculate days in current lifecycle stage
today = pd.Timestamp(datetime.now().date())

model_inventory['Days in Stage'] = (today - model_inventory['Stage Entry Date']).dt.days
model_inventory['Days Since Review'] = (today - model_inventory['Last Review Date']).dt.days
model_inventory['Days Until Review'] = (model_inventory['Next Review Date'] - today).dt.days

print(f"Current Date: {today.strftime('%Y-%m-%d')}\n")
print("Lifecycle Stage Duration:")
print(model_inventory[['Model ID', 'Model Name', 'Lifecycle Stage', 'Days in Stage']])

In [ ]:
def check_approval_gates(model_id: str, inventory_df: pd.DataFrame) -> str:
    """
    Check which approval gates a model has passed based on its approval status.
    
    Expected workflow stages:
    1. Registration
    2. Technical Review
    3. Ethics & Bias Review
    4. Security Review
    5. Business Approval
    6. Production Deployment
    
    Args:
        model_id: The model ID (e.g., 'M001')
        inventory_df: DataFrame with model inventory
    
    Returns:
        Report string with approval gate status and any violations
    """
    # Define the expected approval workflow stages
    approval_workflow_stages = [
        "Registration",
        "Technical Review",
        "Ethics & Bias Review",
        "Security Review",
        "Business Approval",
        "Production Deployment"
    ]
    
    # Get the model from inventory
    model_row = inventory_df[inventory_df['Model ID'] == model_id]
    
    if model_row.empty:
        return f"ERROR: {model_id} not found in inventory"
    
    approval_status = model_row['Approval Status'].values[0]
    lifecycle_stage = model_row['Lifecycle Stage'].values[0]
    
    # Determine which gates have been passed based on approval status
    if approval_status == "Approved":
        gates_passed = approval_workflow_stages  # All gates passed
        report = f"{model_id}: Approved — all gates passed"
    elif approval_status == "In Review":
        gates_passed = approval_workflow_stages[:4]  # Through Security Review
        report = f"{model_id}: In Review — awaiting Business Approval"
    elif approval_status == "Pending":
        gates_passed = approval_workflow_stages[:2]  # Through Technical Review
        report = f"{model_id}: Pending — awaiting Ethics & Bias Review"
    elif approval_status == "Archived":
        gates_passed = approval_workflow_stages  # All gates (historical)
        report = f"{model_id}: Archived — governance complete (Retired model)"
    else:
        gates_passed = []
        report = f"{model_id}: Unknown status ({approval_status})"
    
    # Flag governance violations: Production models with incomplete gates
    if lifecycle_stage == "Production" and approval_status not in ["Approved", "Archived"]:
        report += f" [VIOLATION] Production model missing governance approval!"
    
    return report

# Test the function
print("Approval Gates Status Report:")
print("=" * 80)
for model_id in model_inventory['Model ID']:
    result = check_approval_gates(model_id, model_inventory)
    print(result)
print("=" * 80)

## Part 7: Review Status - Identify Overdue Models

In [ ]:
## Part 8: Lifecycle Summary Dashboard with Visualizations

In [ ]:
# Create dashboard visualizations
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(16, 12))
gs = GridSpec(3, 2, figure=fig, hspace=0.3, wspace=0.3)

# Top row: 2 charts
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])

# Bottom 2 rows: 1 wide chart for timeline
ax3 = fig.add_subplot(gs[1:, :])

fig.suptitle('AI Model Governance Dashboard - Lifecycle & Risk Summary', fontsize=16, fontweight='bold')

# ============================================================================
# Chart 1: Models by Lifecycle Stage (with Retired stage)
# ============================================================================
stage_counts = model_inventory['Lifecycle Stage'].value_counts()
stage_order = ['Development', 'Staging', 'Production', 'Retired']
stage_counts = stage_counts.reindex(stage_order, fill_value=0)

colors_stage = {
    'Development': '#3498db',    # Blue
    'Staging': '#f39c12',         # Orange/Yellow
    'Production': '#2ecc71',      # Green
    'Retired': '#95a5a6'          # Gray
}
colors = [colors_stage.get(stage, '#bdc3c7') for stage in stage_order]

bars1 = ax1.bar(range(len(stage_order)), stage_counts.values, color=colors, edgecolor='black', linewidth=1.5)
ax1.set_xticks(range(len(stage_order)))
ax1.set_xticklabels(stage_order, fontweight='bold')
ax1.set_ylabel('Number of Models', fontweight='bold')
ax1.set_title('Models by Lifecycle Stage', fontweight='bold')
ax1.grid(axis='y', alpha=0.3, linestyle='--')

# Add count labels on bars
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontweight='bold')

# ============================================================================
# Chart 2: Models by Risk Tier
# ============================================================================
risk_counts = model_inventory['Risk Tier'].value_counts()
risk_order = ['Critical', 'High', 'Medium', 'Minimal']
risk_counts = risk_counts.reindex(risk_order, fill_value=0)

colors_risk = {
    'Critical': '#e74c3c',        # Red
    'High': '#e67e22',             # Orange
    'Medium': '#f39c12',           # Yellow
    'Minimal': '#2ecc71'           # Green
}
colors_risk_list = [colors_risk.get(tier, '#bdc3c7') for tier in risk_order]

bars2 = ax2.bar(range(len(risk_order)), risk_counts.values, color=colors_risk_list, edgecolor='black', linewidth=1.5)
ax2.set_xticks(range(len(risk_order)))
ax2.set_xticklabels(risk_order, fontweight='bold')
ax2.set_ylabel('Number of Models', fontweight='bold')
ax2.set_title('Models by Risk Tier', fontweight='bold')
ax2.grid(axis='y', alpha=0.3, linestyle='--')

# Add count labels on bars
for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontweight='bold')

# ============================================================================
# Chart 3: Review Timeline
# ============================================================================
today = pd.Timestamp(datetime.now().date())

# Sort by Model ID for clean visualization
timeline_data = model_inventory.sort_values('Model ID').reset_index(drop=True)
y_positions = range(len(timeline_data))

# Plot timeline for each model
for idx, (i, row) in enumerate(timeline_data.iterrows()):
    last_review = row['Last Review Date']
    next_review = row['Next Review Date']
    is_overdue = next_review < today

    # Draw line from last review to next review
    ax3.plot([last_review, next_review], [idx, idx], 'o-', linewidth=3, markersize=8,
            color='#e74c3c' if is_overdue else '#3498db', alpha=0.7)

    # Add "today" marker
    ax3.axvline(today, color='#2ecc71', linestyle='--', linewidth=2, alpha=0.5)

    # Highlight if overdue (in red)
    if is_overdue:
        ax3.plot(next_review, idx, 'X', color='#e74c3c', markersize=12, markeredgewidth=2, markeredgecolor='#c0392b')

ax3.set_yticks(y_positions)
ax3.set_yticklabels(timeline_data['Model ID'], fontweight='bold')
ax3.set_xlabel('Date', fontweight='bold')
ax3.set_title('Review Timeline: Last Review → Next Review (Red = Overdue)', fontweight='bold')
ax3.grid(axis='x', alpha=0.3, linestyle='--')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498db', edgecolor='black', label='On Track'),
    Patch(facecolor='#e74c3c', edgecolor='black', label='Overdue'),
    plt.Line2D([0], [0], color='#2ecc71', linestyle='--', linewidth=2, label='Today')
]
ax3.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

print("\nVisualization Summary:")
print(f"  Total Models: {len(model_inventory)}")
print(f"  Lifecycle Stages: {dict(stage_counts)}")
print(f"  Risk Tiers: {dict(risk_counts)}")
print(f"  Overdue Reviews: {len(model_inventory[model_inventory['Next Review Date'] < today])}")
print(f"  Current Date: {today.strftime('%Y-%m-%d')}")

## Part 9: Export Model Card to File

## Part 9: Export Model Card to File

In [ ]:
# Save Model Cards to markdown files
import os

output_dir = "model_cards"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

for model_id, model_card_content in model_cards.items():
    filename = f"{output_dir}/model_card_{model_id}.md"
    with open(filename, 'w') as f:
        f.write(model_card_content)
    print(f"✓ Saved {filename}")

# Also save the full inventory as CSV
inventory_export = model_inventory[[
    'Model ID', 'Model Name', 'Version', 'Owner', 'Lifecycle Stage',
    'Approval Status', 'Risk Tier', 'Days Since Review', 'Days Until Review'
]]
inventory_export.to_csv(f"{output_dir}/model_inventory.csv", index=False)
print(f"✓ Saved {output_dir}/model_inventory.csv")

## Summary

This notebook demonstrated:

1. **Model Card Generation**: Created comprehensive Model Cards in Markdown format from structured metadata, including:
   - Model details and ownership
   - Intended use and out-of-scope uses
   - Training data documentation
   - Performance metrics
   - Ethical considerations and bias testing results
   - Limitations and failure modes
   - Recommendations and monitoring plans

2. **Lifecycle Tracking**: Implemented model lifecycle status tracking with pandas, calculating:
   - Days in current lifecycle stage
   - Days since last review
   - Days until next review
   - Identification of overdue models

3. **Governance Insights**: Provided summary statistics and dashboards for:
   - Models by lifecycle stage
   - Models by risk tier
   - Models by approval status
   - Review scheduling and overdue alerts

These tools support RetailGenius' AI model governance program by ensuring comprehensive documentation, regular reviews, and tracking of critical model information.